# Prueba Tecnica - Paulo Hubert

## Strategy outline:

First I conducted some descriptivie analysis and visualization to understand data structure and availability. A few noteworthy points:

1. Data availability is highly different for different SKUs and regions
2. There are many zero-inflated series, with separated waves of non-zero values
3. There's no censored data due to lack of stock, as stocks never fall below 14%
4. Time span is less than a year, so seasonality analysis is not possible

Given the heterogeneity between the different SKUs and regions, and for optimization purposes, we will classify each time series in 3 categories:

1. Low data: series with 5 datapoints or less.
2. Zero-inflated: series with more than 5 datapoints where proportion of 0s is at least 40%
3. Full series: more than 5 datapoints and proportion of zeros lass than 40%

### Goal of this analysis

Define the infrastructure for a model selection engine that will select the best model for each time series (combination of INVENTORY_ID and REGION) and then generate forecasts based on the best model.
The structure should be flexbile, making the incorporation of new models easy, ideally automatically incorporating the new model into the backtest / model selection features.

### Candidate models

1. Naive (mean + standard deviation)
2. ARIMA
3. 

In [1]:
%load_ext autoreload
%autoreload 2

# Data ingestion and descriptive analysis

### Reading the dataset

In [2]:
import pandas as pd

df = pd.read_csv("Technical_test.csv", decimal = ',')
df.head()

,INVENTORY_ID,REGION,DS,SALES,PERC_STOCK
0,SKU-1,FC-BA,2025-11-02,2,0.29
1,SKU-1,FC-PE,2025-11-02,2,0.29
2,SKU-1,FC-PE,2025-11-09,1,0.14
3,SKU-1,FC-PR,2025-11-02,3,0.43
4,SKU-1,FC-RJ,2025-10-26,1,0.14


### Data availability analysis

In [3]:
df['INVENTORY_ID'].value_counts()

SKU-77    144
SKU-76    144
SKU-66    144
SKU-15    144
SKU-53    144
         ... 
SKU-68      1
SKU-34      1
SKU-40      1
SKU-61      1
SKU-64      1
Name: INVENTORY_ID, Length: 78, dtype: int64

In [4]:
# How many classes of data availability are there?
df['INVENTORY_ID'].value_counts().value_counts().sort_index(ascending = False)

144    29
143     2
140     1
135     4
132     2
24      1
22      1
20      2
18      2
16      1
15      2
14      1
12      6
11      2
10      1
9       4
8       1
7       1
6       3
5       3
2       4
1       5
Name: INVENTORY_ID, dtype: int64

In [5]:
# Time span
df['DS'].min(), df['DS'].max()

('2025-07-13', '2025-12-21')

In [6]:
# How many datapoints by series
dfg = df.groupby(['INVENTORY_ID', 'REGION']).agg({'DS' : "nunique"}).reset_index()
dfg.sort_values('DS', inplace = True)
dfg['DS'].value_counts()

24    190
2      50
1      49
3      30
22     22
23     15
5       3
21      3
20      2
7       1
14      1
10      1
12      1
Name: DS, dtype: int64

### Stock analysis: are there any censored datapoints?

In [3]:
df['PERC_STOCK'].describe()

count    5789.000000
mean        0.930301
std         0.183472
min         0.140000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000000
Name: PERC_STOCK, dtype: float64

# Time series classification

In [5]:
from backtest_models import classify_time_series

classified_df = classify_time_series(df)

In [6]:
# Count the unique occurrences of each combination of INVENTORY_ID and REGION for each SERIES_CLASS
classified_df.groupby('SERIES_CLASS')[['INVENTORY_ID', 'REGION']].apply(lambda x: x.drop_duplicates().shape[0])

SERIES_CLASS
Full series      169
Low data         132
Zero-inflated     67
dtype: int64

# Model selection

In [7]:
from backtest_models import backtest_all_series, classify_time_series

th = 1
leave_out_k = 1
conf = 0.95

In [8]:
# Available models by series category
models_by_category = {
    "Low data": ["naive"],
    "Zero-inflated": ["naive", "tsb"],
    "Full series": ["naive", "arima", "tsb"],
}

In [9]:

low_data_df = classified_df[classified_df["SERIES_CLASS"] == "Low data"]

# Right now is doesn't make sense to apply model selection to most any series in this class,
# as there is a single available model.
# Also many series in this class have a single datapoint, rendering model selection impossible
backtest_low_data = backtest_all_series(
    low_data_df,
    th=th,
    leave_out_k=leave_out_k,
    conf=conf,
    model_names=models_by_category["Low data"],
    n_jobs=-1,
)

In [10]:
zero_inflated_df = classified_df[classified_df["SERIES_CLASS"] == "Zero-inflated"]

backtest_zero_inflated = backtest_all_series(
    zero_inflated_df,
    th=th,
    leave_out_k=leave_out_k,
    conf=conf,
    model_names=models_by_category["Zero-inflated"],
    n_jobs=-1,
)

In [ ]:
full_series_df = classified_df[classified_df["SERIES_CLASS"] == "Full series"]

backtest_full_series = backtest_all_series(
    zero_inflated_df,
    th=th,
    leave_out_k=leave_out_k,
    conf=conf,
    model_names=models_by_category["Full series"],
    n_jobs=-1,
)

In [ ]:
def flatten_point_results(results_dict, series_class):
    rows = []
    for inventory_id, regions_dict in results_dict.items():
        for region, payload in regions_dict.items():
            if payload is None:
                continue
            point_results = payload.get("point_results", {})
            for model_name, metrics in point_results.items():
                rows.append(
                    {
                        "INVENTORY_ID": inventory_id,
                        "REGION": region,
                        "SERIES_CLASS": series_class,
                        "MODEL": model_name,
                        "MAE": metrics.get("MAE"),
                        "RMSE": metrics.get("RMSE"),
                        "MAPE": metrics.get("MAPE"),
                        "MASE": metrics.get("MASE"),
                    }
                )
    return rows

point_rows = []
point_rows += flatten_point_results(backtest_low_data, "Low data")
point_rows += flatten_point_results(backtest_zero_inflated, "Zero-inflated")
point_rows += flatten_point_results(backtest_full_series, "Full series")

final_point_metrics_df = pd.DataFrame(point_rows)
final_point_metrics_df.sort_values(
    ["INVENTORY_ID", "REGION", "SERIES_CLASS", "MODEL"], inplace=True
)
final_point_metrics_df.reset_index(drop=True, inplace=True)
final_point_metrics_df

NameError: name 'backtest_full_series' is not defined